<a href="https://colab.research.google.com/github/derekmok/machine-vision-coursework/blob/main/Machine_Vision_Final_Lab_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git init .
!git remote add origin https://github.com/derekmok/machine-vision-coursework.git
!git pull origin main

!pip install -r requirements.txt

In [1]:
# Here is a basic implementation of the above two TODOs. You can assume the first TODO is completed correctly.

# Please modify this code to suit you best, as you decide on your preferred model architecture.

# For example, below here we are padding every video to 1,000 frames. That may or may not be a good idea.

from data_loader import VideoDataset

# Create a Model

For this assignment, we request you use PyTorch. Below is an example of how to instantiate a very basic PyTorch model.

Note, this model below needs a _lot_ of work.

Please include your code for creating your model below.

The only constraint here is that you define a Python object which inherits from a PyTorch nn.Module object. Beyond that, please feel free to implement anything you like: Transformer, Vision Transformer, MLP, CNN, etc.

### TODO 4

Create your model.

In [2]:
import torch
from neural_net.temporal_conv_net import TCNPushUpCounter

# Train your Model

### TODO 5

Training time! Please include your training code below.

As per above, please feel free (and encouraged) to rip out all of the below code and replace with your (much better) code.

The below should just be used as an example to get you started.

In [3]:
import torch.optim as optim
from neural_net.ensemble_trainer import EnsembleTrainer
from feature_engineering.transforms import Compose, RandomScaling, RandomNoise, RandomTimeWarp, RandomSequenceReverse, \
    RandomSequenceRepeat, RandomHorizontalFlipLandmarks, RandomDropout
from neural_net.kl_divergence_loss import KLDivergenceDensityLoss
from neural_net.loocv_trainer import LOOCVTrainer


def train_model():
    trainer = LOOCVTrainer(
        model_factory=lambda: TCNPushUpCounter(),
        loss_fn=KLDivergenceDensityLoss(lambda_count=1e-1),
        optimizer_factory=lambda parameters : optim.AdamW(parameters),
        patience=100,
        max_epochs=1000,
    )

    return trainer.train(
        dataset=VideoDataset("video-data"),
        train_transform=Compose([
            RandomSequenceRepeat(),
            RandomSequenceReverse(),
            RandomHorizontalFlipLandmarks(),
            RandomTimeWarp(p=0.8),
            RandomScaling(),
            RandomNoise(p=1.0),
            RandomDropout()
        ])
    )

In [4]:
# setting a manual seed for reproducibility
!mkdir -p .models && wget --no-clobber -O .models/pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task
torch.manual_seed(42)
training_results = train_model()

File ‘.models/pose_landmarker.task’ already there; not retrieving.

Starting Leave One Out Cross Validation with 77 folds
Training fold 1/77...


KeyboardInterrupt: 

# Evaluation

## TODO 6

Include any code which you feel is useful for evaluating your model performance below.